# 7. Cohort Study Sub-analysis Data Processing
In this notebook, we prepare data for the cohort study sub-analysis.

In [ ]:
library(ggplot2)
library(tidyverse)
library(readxl)

Warning message:
"package 'tidyverse' was built under R version 4.3.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
base_dir = getwd() #Where the notebook is in
setwd("..")
setwd("..")
index_dir = getwd() #Where the review index is in
setwd(base_dir)
data_dir = file.path(index_dir, "Data") #Where the extracted Excel data files are
output_dir = file.path(base_dir, "Model Inputs") #Where we will output the processed data to be used as Stan input

save_output = TRUE
min_cases = 1

In [ ]:
#Regions part of the analysis
regions = c("Americas", "Asia")
remove_regions = c("Africa")

#Reference class for the logistic regression
ref_region = "Asia"
ref_serotype_exposure = "Secondary-DENV2"
ref_scenario = paste0(ref_region, "-", ref_serotype_exposure)

#Generate combinations of prior exposure and serotype
serotype_exposure = expand.grid(Serotype = paste0("DENV", 1:4), PriorExp = c("Primary", "Secondary")) %>%
                        mutate(SeroPriorExp = paste0(PriorExp, "-", Serotype)) %>% mutate(ColIndex = 1:nrow(.))

## Data Reading
In this section, we read in the index and the data from the Excel files.

In [ ]:
index_df = read_excel(file.path(index_dir, "ReviewIndex_Final.xlsx"), sheet = "Main")
reg_mapper = read_excel(file.path(index_dir, "ReviewIndex_Final.xlsx"), sheet = "CountryRegions")

#Get only the included studies
include_df = index_df %>% filter(FinalDecision %in% c("Include"))

#If the CovidenceID is blank, use the CovidenceID_JanUpdate value
include_df = include_df %>% mutate(CovidenceID = ifelse(is.na(CovidenceID), CovidenceID_JanUpdate, CovidenceID))

#Create a DataFrame with the filename pointing to the Excel file with the data
filename_df = include_df %>% select(ID, CovidenceID, Name, Country, SpecificLocation, WHOClass, InfectionTypes) %>% 
                mutate(File = paste0(ID, "-", Name, ".xlsx")) %>%
                merge(reg_mapper, by = "Country") #Label the study with the region the study country is in

Here we filter things to only the cohort studies of interest by using the CovidenceIDs 
Currently, there are two in three studies of interest: 
1. Narvaez et al. 2025 in Nicaragua
2. Fried et al. 2010 in Thailand (QNSICH)
3. Stephens et al. 2002 in Thailand (seems to be both QSNICH and KPPH based on cited studies and acknowledgements)

In [ ]:
cohort_ids = c("#4544 - Stephens 2002", "#1583 - Fried 2010", "#66694 - Narvaez 2025")
filename_df_filt = filename_df %>% filter(CovidenceID %in% cohort_ids)

In [ ]:
#Function to read in the data
read_file = function(curr_row){
    #Filename of the Excel file to read
    curr_filename = curr_row["File"]
    
    #Vector of prior exposure statuses given by the study, which correspond with Excel sheets to read
    curr_sheets = curr_row["InfectionTypes"] %>% str_split(pattern = ", ") %>% unlist
    
    #Check for excluded sheets, and print them out as a warning
    for(temp in curr_sheets){
        if(!temp %in% c("Unspecified", "Primary", "Secondary", "ND")){
            print(temp)
        }
    }
    
    #We exclude sheets outside Unspecified, Primary, Secondary, and ND
    curr_sheets = curr_sheets[curr_sheets %in% c("Unspecified", "Primary", "Secondary", "ND")]
    
    
    #Start reading sheet
    curr_sheet = curr_sheets[1]
    read_sheet = function(curr_sheet){
        curr_data = suppressMessages({read_excel(file.path(data_dir, curr_filename), sheet = curr_sheet) %>% rename(Serotype = `...1`)})
        
        #Check which severity class type the study is in: (1) 1997-type, (2) 2009-type, (3) Hospitalisation
        class_type = ifelse("Hospitalised" %in% colnames(curr_data), "Hospitalisation", ifelse("DF" %in% colnames(curr_data), "1997-type", "2009-type"))
        
        #Check if the study has any asymptomatic cases extracted and print a message if it does
        asymp_checker = curr_data %>% pivot_longer(-Serotype, names_to = "SeverityClass", values_to = "Count") %>% 
                        filter((SeverityClass == "Asymptomatic") & (Count > 0))
        if(nrow(asymp_checker) > 0){print(paste0("Asymptomatic in ", curr_filename))}
        
        #Format the data
        suppressMessages({curr_data = curr_data %>% pivot_longer(-Serotype, names_to = "SeverityClass", values_to = "Count") %>%
                    filter(SeverityClass != "Asymptomatic") %>% #Remove asymptomatic cases
                    mutate(Severity = ifelse(SeverityClass %in% c("DHF", "DSS", "DHF/DSS", "Hospitalised", "SD"), "Severe", "NonSevere")) %>%#Label cases as severe or non-severe
                    select(-SeverityClass) %>% replace_na(list(Count = 0)) %>% #Change all the NAs as 0s
                    group_by(Serotype, Severity) %>% summarise(Count = sum(Count)) %>% ungroup %>% #Some counts of severe and non-severe
                    pivot_wider(values_from = Count, names_from = Severity) %>% #Pivot to have separate severe, non-severe columns
                    mutate(N = Severe + NonSevere, ID = curr_row["ID"], CovidenceID = curr_row["CovidenceID"],
                          PriorExp = curr_sheet, SeveritySystem = curr_row["WHOClass"],
                          SeverityType = class_type, Region = curr_row["Region"], Name = curr_filename)}) #Label the row of data
        return(curr_data)
    }
    #Read all the sheets of the current file
    to_ret = do.call(rbind, lapply(curr_sheets, read_sheet))
    return(to_ret)
}

In [ ]:
data_set = do.call(rbind, apply(filename_df_filt, 1, read_file)) %>%
            filter(SeverityType == "1997-type") %>%
            mutate(SeroPriorExp = paste0(PriorExp, "-", Serotype)) %>%
            filter(N >= min_cases)

In [ ]:
data_set

Serotype,NonSevere,Severe,N,ID,CovidenceID,PriorExp,SeveritySystem,SeverityType,Region,Name,SeroPriorExp
<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DENV1,262,15,277,0331,#66694 - Narvaez 2025,Primary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Primary-DENV1
DENV2,164,14,178,0331,#66694 - Narvaez 2025,Primary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Primary-DENV2
DENV3,331,56,387,0331,#66694 - Narvaez 2025,Primary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Primary-DENV3
DENV4,57,0,57,0331,#66694 - Narvaez 2025,Primary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Primary-DENV4
DENV1,239,27,266,0331,#66694 - Narvaez 2025,Secondary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Secondary-DENV1
DENV2,610,207,817,0331,#66694 - Narvaez 2025,Secondary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Secondary-DENV2
DENV3,282,88,370,0331,#66694 - Narvaez 2025,Secondary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Secondary-DENV3
DENV4,249,9,258,0331,#66694 - Narvaez 2025,Secondary,1997,1997-type,Americas,0331-Narvaez_2025a.xlsx,Secondary-DENV4
DENV1,21,4,25,0158,#4544 - Stephens 2002,Primary,1997Binary,1997-type,Asia,0158-Stephens_2002.xlsx,Primary-DENV1


## Creating Stan Input
In this section, we then generate input for the Stan logistic regression model. Compared to the main analysis, this would be a much simpler model excluding the effect of geographic region and removing the need to fit a correlation matrix. We then need to compute the following
1. n_row (int): Number of studies included in the analysis, each will be represented as a row in the total/severe matrix.
2. n_col (int): Number of serotype-prior exposure status combinations, which will be the number of columns in the total/severe matrix.
3. n_outcomes (int): Number of outcomes in the analysis i.e. number of non-zero cells in the total matrix
4. total (matrix; n_row x n_col): Number of dengue cases for study in row $i$ with serotype-prior exposure in col $j$
5. severe (matrix; n_row x n_col): Number of severe cases for study in row $i$ with serotype-prior exposure in col $j$
6. row_indices (array int; n_outcomes): Row index for each outcome included in the analysis
7. col_indices (array int; n_outcomes): Column index for each outcome included in the analysis
8. num_results (array int; n_row): Number of outcomes for study index $i$ (i.e. non-zero rows in row $i$ of the total matrix)
9. n_sero_prior (int): Number of serotype-prior exposure combinations included in the analysis
10. char_matrix (array int; n_outcomes x (n_sero_prior - 1)): Characteristic matrix for each outcome included in the analysis. This will be multiplied to beta values to then get effects to add to the intercept.

In [ ]:
n_row = data_set %>% pull(CovidenceID) %>% unique %>% length
n_col = data_set %>% pull(SeroPriorExp) %>% unique %>% length
n_outcomes = nrow(data_set)

total = array(0, dim = c(n_row, n_col))
severe = array(0, dim = c(n_row, n_col))
row_indices = array(0, dim = n_outcomes)
col_indices = array(0, dim = n_outcomes)

study_df = data_set %>% select(CovidenceID) %>% distinct %>% mutate(RowIndex = 1:nrow(.))
scenario_df = data_set %>% select(SeroPriorExp) %>% distinct %>% mutate(ColIndex = 1:nrow(.))
outcome_df = data_set %>% select(CovidenceID, Serotype, PriorExp, SeroPriorExp, Severe, N) %>% 
            left_join(study_df, by = "CovidenceID") %>% 
            left_join(scenario_df, by = "SeroPriorExp")

for(i in 1:nrow(outcome_df)){
    curr_row = outcome_df[i, ]
    curr_row_ind = curr_row %>% pull(RowIndex)
    curr_col_ind= curr_row %>% pull(ColIndex)

    curr_total = curr_row %>% pull(N)
    curr_severe = curr_row %>% pull(Severe)

    #Set the values in the total and severe matrix
    total[curr_row_ind, curr_col_ind] = curr_total
    severe[curr_row_ind, curr_col_ind] = curr_severe
    
    #Set index values
    row_indices[i] = curr_row_ind
    col_indices[i] = curr_col_ind
}


outcome_counter = outcome_df %>% select(CovidenceID, SeroPriorExp) %>% group_by(CovidenceID) %>% summarise(OutcomeCount = n_distinct(SeroPriorExp))
num_results = outcome_counter %>% pull(OutcomeCount)
n_sero_prior = nrow(scenario_df)

#We then form the char_matrix based on the reference class
char_matrix = array(0, dim = c(n_outcomes, dim = n_sero_prior))
char_matrix[,1] = 1

#Reference class
ref_serotype_exposure = "Secondary-DENV2"

#Create a DataFrame with indices for filling the characteristic matrix
char_mat_guide = scenario_df %>% select(SeroPriorExp)
ref_ind = which(char_mat_guide$SeroPriorExp == ref_serotype_exposure) #Find which position the reference class belongs to
char_mat_ind = 1:(n_sero_prior-1) %>% append(-1, after = (ref_ind -1)) #We set to -1 the reference class, while the other classes range from 1:n_sero_prior
char_mat_guide = char_mat_guide %>% mutate(CharMatIndex = char_mat_ind) 

for(i in 1:nrow(outcome_df)){
    curr_row = outcome_df[i,]
    curr_SeroPriorExp = curr_row %>% pull(SeroPriorExp)
    char_mat_SeroPriorExp_ind = char_mat_guide %>% filter(SeroPriorExp == curr_SeroPriorExp) %>% pull(CharMatIndex) #The index we should set in the char matrix
    if(char_mat_SeroPriorExp_ind != -1){
        #Have to offset by 1 due to the intercept column at the start
        char_matrix[i, char_mat_SeroPriorExp_ind + 1] = 1
    }
}

In [ ]:
data_list = list(
    n_row = n_row, 
    n_col = n_col,
    n_outcomes = n_outcomes, 
    total = total,
    severe = severe,
    row_indices = row_indices,
    col_indices = col_indices,
    num_results = num_results,
    n_sero_prior = n_sero_prior, 
    char_matrix = char_matrix
)
scenario_df = scenario_df %>% left_join(char_mat_guide, by = "SeroPriorExp") #Add the char matrix indices to the scenario_df

return_list = list(data_list = data_list,
                  outcome_df = outcome_df,
                  study_df = study_df,
                  scenario_df = scenario_df,
                  char_mat_guide = char_mat_guide,
                  ref_serotype_exposure = ref_serotype_exposure)

In [ ]:
#Create an RDS file that contains what we need for Stan input
if(save_output){
    saveRDS(return_list, file.path(output_dir, "cohort_input_no_min_cases.rds"))
}